In [9]:
import requests
import time
from IPython.display import clear_output

# ESP32 IP-Adresse
ESP32_IP = "192.168.178.112"
BASE_URL = f"http://{ESP32_IP}"

print(f"✅ Verbinde mit ESP32 auf {BASE_URL}")
print(f"Testing connection...")

try:
    response = requests.get(BASE_URL, timeout=3)
    if response.status_code == 200:
        print("✅ Verbindung erfolgreich!")
    else:
        print(f"⚠️ Verbindung OK, aber Status Code: {response.status_code}")
except Exception as e:
    print(f"❌ Fehler bei der Verbindung: {e}")

✅ Verbinde mit ESP32 auf http://192.168.178.112
Testing connection...
✅ Verbindung erfolgreich!


## 🎯 Servo Control (0-90 Grad)

API Endpoint: `/setServo?angle=<0-180>`

Der Servo wird in verschiedene Positionen zwischen 0 und 90 Grad bewegt.

In [10]:
def set_servo_angle(angle):
    """Setzt den Servo auf einen bestimmten Winkel (0-180 Grad)"""
    url = f"{BASE_URL}/setServo?angle={angle}"
    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            print(f"✅ Servo auf {angle}° gesetzt: {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

# Test: Servo von 0 bis 90 Grad in 30-Grad-Schritten
print("🎯 Servo Test: 0° → 30° → 60° → 90° → 0°\n")

angles = [0, 30, 60, 90, 60, 30, 0]
for angle in angles:
    set_servo_angle(angle)
    time.sleep(0.8)  # Warte 800ms zwischen Bewegungen

print("\n✅ Servo Test abgeschlossen!")

🎯 Servo Test: 0° → 30° → 60° → 90° → 0°

✅ Servo auf 0° gesetzt: Servo positioned to 0 degrees
✅ Servo auf 30° gesetzt: Servo positioned to 30 degrees
✅ Servo auf 30° gesetzt: Servo positioned to 30 degrees
✅ Servo auf 60° gesetzt: Servo positioned to 60 degrees
✅ Servo auf 60° gesetzt: Servo positioned to 60 degrees
✅ Servo auf 90° gesetzt: Servo positioned to 90 degrees
✅ Servo auf 90° gesetzt: Servo positioned to 90 degrees
✅ Servo auf 60° gesetzt: Servo positioned to 60 degrees
✅ Servo auf 60° gesetzt: Servo positioned to 60 degrees
✅ Servo auf 30° gesetzt: Servo positioned to 30 degrees
✅ Servo auf 30° gesetzt: Servo positioned to 30 degrees
✅ Servo auf 0° gesetzt: Servo positioned to 0 degrees
✅ Servo auf 0° gesetzt: Servo positioned to 0 degrees

✅ Servo Test abgeschlossen!

✅ Servo Test abgeschlossen!


## 🔄 28BYJ-48 Stepper Motor Control

API Endpoint: `/motor28byj48?action=moveRelative&steps=<anzahl>&direction=<1/-1>&speed=<0-100>`

- `steps`: Anzahl der Schritte (absoluter Wert)
- `direction`: 1 = vorwärts (➡️), -1 = rückwärts (⬅️)
- `speed`: Geschwindigkeit 0-100%

In [11]:
def move_28byj48_relative(steps, direction, speed=50):
    """
    Bewegt den 28BYJ-48 Motor relativ
    
    Args:
        steps: Anzahl der Schritte (positiv)
        direction: 1 für vorwärts, -1 für rückwärts
        speed: Geschwindigkeit 0-100%
    """
    url = f"{BASE_URL}/motor28byj48?action=moveRelative&steps={steps}&direction={direction}&speed={speed}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            direction_symbol = "➡️" if direction > 0 else "⬅️"
            print(f"✅ Motor bewegt {direction_symbol} {steps} Schritte @ {speed}%: {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

# Test: Motor hin und her bewegen
print("🔄 28BYJ-48 Motor Test\n")

movements = [
    (100, 1, 50),   # 100 Schritte vorwärts, 50% Speed
    (50, -1, 60),   # 50 Schritte rückwärts, 60% Speed
    (200, 1, 70),   # 200 Schritte vorwärts, 70% Speed
    (150, -1, 50),  # 150 Schritte rückwärts, 50% Speed
]

for steps, direction, speed in movements:
    move_28byj48_relative(steps, direction, speed)
    time.sleep(2)  # Warte bis Bewegung abgeschlossen ist

print("\n✅ Motor Test abgeschlossen!")

🔄 28BYJ-48 Motor Test

✅ Motor bewegt ➡️ 100 Schritte @ 50%: Moving 100 steps
✅ Motor bewegt ➡️ 100 Schritte @ 50%: Moving 100 steps
✅ Motor bewegt ⬅️ 50 Schritte @ 60%: Moving -50 steps
✅ Motor bewegt ⬅️ 50 Schritte @ 60%: Moving -50 steps
✅ Motor bewegt ➡️ 200 Schritte @ 70%: Moving 200 steps
✅ Motor bewegt ➡️ 200 Schritte @ 70%: Moving 200 steps
✅ Motor bewegt ⬅️ 150 Schritte @ 50%: Moving -150 steps
✅ Motor bewegt ⬅️ 150 Schritte @ 50%: Moving -150 steps

✅ Motor Test abgeschlossen!

✅ Motor Test abgeschlossen!


## 💡 LED Control

### API Endpoints:
- `/color?index=<0-6>` - Vordefinierte Farben (0=Rot, 1=Grün, 2=Blau, 3=Gelb, 4=Lila, 5=Orange, 6=Weiß)
- `/hexcolor?hex=<#RRGGBB>` - Benutzerdefinierte Farbe
- `/setBrightness?value=<0-255>` - Helligkeit einstellen

In [12]:
def set_led_color_index(color_index):
    """Setzt LED auf vordefinierte Farbe (0-6)"""
    color_names = ["Rot", "Grün", "Blau", "Gelb", "Lila", "Orange", "Weiß"]
    url = f"{BASE_URL}/color?index={color_index}"
    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            color_name = color_names[color_index] if 0 <= color_index < len(color_names) else "Unbekannt"
            print(f"✅ LED Farbe: {color_name} - {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def set_led_hex_color(hex_color):
    """Setzt LED auf benutzerdefinierte Hex-Farbe (z.B. #FF00FF)"""
    url = f"{BASE_URL}/hexcolor?hex={hex_color}"
    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            print(f"✅ LED Farbe: {hex_color} - {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def set_led_brightness(brightness):
    """Setzt LED Helligkeit (0-255)"""
    url = f"{BASE_URL}/setBrightness?value={brightness}"
    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            print(f"✅ LED Helligkeit: {brightness} - {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

# Test: LED mit verschiedenen Farben durchlaufen
print("💡 LED Test - Vordefinierte Farben\n")

for i in range(7):
    set_led_color_index(i)
    time.sleep(0.5)

print("\n✅ LED Farben Test abgeschlossen!")

💡 LED Test - Vordefinierte Farben

✅ LED Farbe: Rot - Color changed to index 0
✅ LED Farbe: Grün - Color changed to index 1
✅ LED Farbe: Grün - Color changed to index 1
✅ LED Farbe: Blau - Color changed to index 2
✅ LED Farbe: Blau - Color changed to index 2
✅ LED Farbe: Gelb - Color changed to index 3
✅ LED Farbe: Gelb - Color changed to index 3
✅ LED Farbe: Lila - Color changed to index 4
✅ LED Farbe: Lila - Color changed to index 4
✅ LED Farbe: Orange - Color changed to index 5
✅ LED Farbe: Orange - Color changed to index 5
✅ LED Farbe: Weiß - Color changed to index 6
✅ LED Farbe: Weiß - Color changed to index 6

✅ LED Farben Test abgeschlossen!

✅ LED Farben Test abgeschlossen!


In [13]:
# Test: LED mit benutzerdefinierten Hex-Farben
print("💡 LED Test - Benutzerdefinierte Farben\n")

custom_colors = [
    "#FF0000",  # Rot
    "#00FF00",  # Grün
    "#0000FF",  # Blau
    "#FF00FF",  # Magenta
    "#00FFFF",  # Cyan
    "#FFFF00",  # Gelb
]

for color in custom_colors:
    set_led_hex_color(color)
    time.sleep(0.5)

print("\n✅ LED Custom Farben Test abgeschlossen!")

💡 LED Test - Benutzerdefinierte Farben

✅ LED Farbe: #FF0000 - Color changed to #
✅ LED Farbe: #00FF00 - Color changed to #
✅ LED Farbe: #00FF00 - Color changed to #
✅ LED Farbe: #0000FF - Color changed to #
✅ LED Farbe: #0000FF - Color changed to #
✅ LED Farbe: #FF00FF - Color changed to #
✅ LED Farbe: #FF00FF - Color changed to #
✅ LED Farbe: #00FFFF - Color changed to #
✅ LED Farbe: #00FFFF - Color changed to #
✅ LED Farbe: #FFFF00 - Color changed to #
✅ LED Farbe: #FFFF00 - Color changed to #

✅ LED Custom Farben Test abgeschlossen!

✅ LED Custom Farben Test abgeschlossen!


In [14]:
# Test: LED Helligkeit von dunkel bis hell
print("💡 LED Test - Helligkeit\n")

# Setze Farbe auf Weiß
set_led_color_index(6)
time.sleep(0.3)

# Helligkeit hochfahren
brightness_levels = [10, 50, 100, 150, 200, 255]
for brightness in brightness_levels:
    set_led_brightness(brightness)
    time.sleep(0.5)

# Zurück auf mittlere Helligkeit
set_led_brightness(50)

print("\n✅ LED Helligkeit Test abgeschlossen!")

💡 LED Test - Helligkeit

✅ LED Farbe: Weiß - Color changed to index 6
✅ LED Helligkeit: 10 - Brightness set to 10
✅ LED Helligkeit: 10 - Brightness set to 10
✅ LED Helligkeit: 50 - Brightness set to 50
✅ LED Helligkeit: 50 - Brightness set to 50
✅ LED Helligkeit: 100 - Brightness set to 100
✅ LED Helligkeit: 100 - Brightness set to 100
✅ LED Helligkeit: 150 - Brightness set to 150
✅ LED Helligkeit: 150 - Brightness set to 150
✅ LED Helligkeit: 200 - Brightness set to 200
✅ LED Helligkeit: 200 - Brightness set to 200
✅ LED Helligkeit: 255 - Brightness set to 255
✅ LED Helligkeit: 255 - Brightness set to 255
✅ LED Helligkeit: 50 - Brightness set to 50

✅ LED Helligkeit Test abgeschlossen!
✅ LED Helligkeit: 50 - Brightness set to 50

✅ LED Helligkeit Test abgeschlossen!


## 🎬 Komplette Demo-Sequenz

Eine koordinierte Demo mit allen Komponenten gleichzeitig.

In [15]:
print("🎬 Starte komplette Demo-Sequenz\n")
print("=" * 50)

# Schritt 1: LED auf Rot, Servo auf 0°
print("\n🎬 Schritt 1: Initialisierung")
set_led_color_index(0)  # Rot
set_servo_angle(0)
time.sleep(1)

# Schritt 2: Motor vorwärts, Servo auf 45°, LED Grün
print("\n🎬 Schritt 2: Motor vorwärts, Servo 45°, LED Grün")
move_28byj48_relative(150, 1, 60)
time.sleep(0.5)
set_servo_angle(45)
set_led_color_index(1)  # Grün
time.sleep(2)

# Schritt 3: Motor rückwärts, Servo auf 90°, LED Blau
print("\n🎬 Schritt 3: Motor rückwärts, Servo 90°, LED Blau")
move_28byj48_relative(100, -1, 70)
time.sleep(0.5)
set_servo_angle(90)
set_led_color_index(2)  # Blau
time.sleep(2)

# Schritt 4: Motor vorwärts, Servo zurück auf 0°, LED Gelb
print("\n🎬 Schritt 4: Motor vorwärts, Servo 0°, LED Gelb")
move_28byj48_relative(200, 1, 50)
time.sleep(0.5)
set_servo_angle(0)
set_led_color_index(3)  # Gelb
time.sleep(2)

# Schritt 5: Finale - Alles zurück zu Start, LED Weiß
print("\n🎬 Schritt 5: Rückkehr zur Ausgangsposition")
move_28byj48_relative(250, -1, 60)
time.sleep(0.5)
set_servo_angle(45)
set_led_color_index(6)  # Weiß
time.sleep(2)

print("\n" + "=" * 50)
print("✅ Demo-Sequenz abgeschlossen!")

🎬 Starte komplette Demo-Sequenz


🎬 Schritt 1: Initialisierung
✅ LED Farbe: Rot - Color changed to index 0
✅ Servo auf 0° gesetzt: Servo positioned to 0 degrees

🎬 Schritt 2: Motor vorwärts, Servo 45°, LED Grün

🎬 Schritt 2: Motor vorwärts, Servo 45°, LED Grün
✅ Motor bewegt ➡️ 150 Schritte @ 60%: Moving 150 steps
✅ Motor bewegt ➡️ 150 Schritte @ 60%: Moving 150 steps
✅ Servo auf 45° gesetzt: Servo positioned to 45 degrees
✅ LED Farbe: Grün - Color changed to index 1
✅ Servo auf 45° gesetzt: Servo positioned to 45 degrees
✅ LED Farbe: Grün - Color changed to index 1

🎬 Schritt 3: Motor rückwärts, Servo 90°, LED Blau

🎬 Schritt 3: Motor rückwärts, Servo 90°, LED Blau
✅ Motor bewegt ⬅️ 100 Schritte @ 70%: Moving -100 steps
✅ Motor bewegt ⬅️ 100 Schritte @ 70%: Moving -100 steps
✅ Servo auf 90° gesetzt: Servo positioned to 90 degrees
✅ LED Farbe: Blau - Color changed to index 2
✅ Servo auf 90° gesetzt: Servo positioned to 90 degrees
✅ LED Farbe: Blau - Color changed to index 2

🎬 Schritt 

## 📊 Status Abfrage

Optional: Motor-Position und andere Status-Informationen abrufen

In [16]:
def get_motor_status():
    """Holt den aktuellen Motor-Status"""
    url = f"{BASE_URL}/motor28byj48?action=getStatus"
    try:
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            print(f"📊 Motor Status:\n{response.text}")
            return response.text
        else:
            print(f"⚠️ Status Code: {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Fehler: {e}")
        return None

# Status abrufen
get_motor_status()

⚠️ Status Code: 400


## 🔧 Erweiterte Funktionen

Weitere API-Endpunkte die verfügbar sind:

### 28BYJ-48 Motor:
- `/motor28byj48?action=moveToPosition&position=<pos>&speed=<speed>` - Absolute Position (in Steps)
- `/motor28byj48?action=home` - Zur Home-Position fahren
- `/motor28byj48?action=calibrate` - Motor kalibrieren (Position auf 0 setzen)

**Hinweis:** Der 28BYJ-48 ist ein Stepper-Motor mit Getriebe. Die Bewegung sollte in **Steps** angegeben werden, nicht in Grad, da das Getriebe-Verhältnis variieren kann.

### Pin-Konfiguration:
- `/getPinConfig` - Aktuelle Pin-Konfiguration abrufen
- `/motor28byj48?action=setPins&pin1=<p1>&pin2=<p2>&pin3=<p3>&pin4=<p4>` - Pins neu zuweisen

## ⚙️ Pin-Konfiguration setzen

Die GPIO-Pins für alle Komponenten können über die API neu konfiguriert und im EEPROM gespeichert werden.

In [17]:
def set_28byj48_pins(pin1, pin2, pin3, pin4):
    """
    Setzt die GPIO-Pins für den 28BYJ-48 Motor und speichert sie im EEPROM
    
    Args:
        pin1-pin4: GPIO Pins für IN1-IN4 des Motors
    """
    url = f"{BASE_URL}/motor28byj48?action=setPins&pin1={pin1}&pin2={pin2}&pin3={pin3}&pin4={pin4}"
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print(f"✅ 28BYJ-48 Pins gesetzt: IN1={pin1}, IN2={pin2}, IN3={pin3}, IN4={pin4}")
            print(f"   {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def set_servo_pin(pin):
    """
    Setzt den GPIO-Pin für den Servo und speichert ihn im EEPROM
    
    Args:
        pin: GPIO Pin für den Servo
    """
    url = f"{BASE_URL}/setServoPin?pin={pin}"
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print(f"✅ Servo Pin gesetzt: GPIO {pin}")
            print(f"   {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def set_led_pin(pin):
    """
    Setzt den GPIO-Pin für die LED und speichert ihn im EEPROM
    
    Args:
        pin: GPIO Pin für die LED
    """
    url = f"{BASE_URL}/setLedPin?pin={pin}"
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print(f"✅ LED Pin gesetzt: GPIO {pin}")
            print(f"   {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def set_button_pin(pin):
    """
    Setzt den GPIO-Pin für den Button und speichert ihn im EEPROM
    
    Args:
        pin: GPIO Pin für den Button
    """
    url = f"{BASE_URL}/setButtonPin?pin={pin}"
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print(f"✅ Button Pin gesetzt: GPIO {pin}")
            print(f"   {response.text}")
            return True
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return False

def get_pin_config():
    """Holt die aktuelle Pin-Konfiguration"""
    url = f"{BASE_URL}/getPinConfig"
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print(f"📋 Aktuelle Pin-Konfiguration:\n{response.text}")
            return response.text
        else:
            print(f"❌ Fehler: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Verbindungsfehler: {e}")
        return None

print("⚙️ Pin-Konfiguration Funktionen geladen")

⚙️ Pin-Konfiguration Funktionen geladen


### Aktuelle Pin-Konfiguration abrufen

In [18]:
# Aktuelle Pin-Konfiguration anzeigen
get_pin_config()

❌ Fehler: Status 404


### Standard-Pins setzen

Konfiguriert alle GPIO-Pins auf die Standard-Werte und speichert sie im EEPROM.

In [19]:
print("⚙️ Setze Standard Pin-Konfiguration\n")
print("=" * 50)

# 28BYJ-48 Motor Pins (IN1, IN2, IN3, IN4)
print("\n🔄 28BYJ-48 Stepper Motor:")
set_28byj48_pins(19, 20, 21, 47)
time.sleep(0.5)

# Servo Pin
print("\n🎯 Servo Motor:")
set_servo_pin(14)
time.sleep(0.5)

# LED Pin
print("\n💡 LED:")
set_led_pin(38)
time.sleep(0.5)

# Button Pin
print("\n🔘 Button:")
set_button_pin(45)
time.sleep(0.5)

print("\n" + "=" * 50)
print("✅ Alle Pins konfiguriert und im EEPROM gespeichert!")
print("\n💾 Die Konfiguration bleibt nach einem Neustart erhalten.")

⚙️ Setze Standard Pin-Konfiguration


🔄 28BYJ-48 Stepper Motor:
✅ 28BYJ-48 Pins gesetzt: IN1=19, IN2=20, IN3=21, IN4=47
   Pins configured and saved to EEPROM: 19, 20, 21, 47

🎯 Servo Motor:
❌ Fehler: Status 404

💡 LED:
❌ Fehler: Status 404

🔘 Button:
❌ Fehler: Status 404

✅ Alle Pins konfiguriert und im EEPROM gespeichert!

💾 Die Konfiguration bleibt nach einem Neustart erhalten.


### Benutzerdefinierte Pins setzen

Beispiel: Andere GPIO-Pins verwenden

In [ ]:
# Beispiel: Alternative Pin-Konfiguration
print("🔧 Beispiel: Alternative GPIO-Pins setzen\n")

# Ändere nur den Servo Pin als Beispiel
print("Ändere Servo Pin von 14 auf 15:")
set_servo_pin(15)
time.sleep(1)

# Zurück zum Standard
print("\nZurück zum Standard Pin 14:")
set_servo_pin(14)

print("\n✅ Pin-Änderung demonstriert!")
print("ℹ️  Hinweis: Pins können jederzeit über die API geändert werden")

### Verifizierung nach Pin-Änderung

Nach dem Ändern der Pins sollte die Konfiguration überprüft werden.

In [ ]:
# Verifiziere die aktuelle Pin-Konfiguration
print("📋 Verifizierung der Pin-Konfiguration:\n")
get_pin_config()

print("\n" + "=" * 50)
print("✅ Pin-Konfiguration verifiziert!")
print("\nℹ️  Diese Einstellungen sind:")
print("   • Im EEPROM gespeichert")
print("   • Bleiben nach Neustart erhalten")
print("   • Können jederzeit über die API geändert werden")